# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [11]:
# imports

import os
import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


In [35]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'



In [12]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

if api_key and api_key.startswith("sk-or-"):
    print("api key VALID")
else:
    print("api key INVALID!")

openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

api key VALID


In [32]:
requests.get("http://localhost:11434").content


b'Ollama is running'

In [33]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [36]:
response = ollama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = [{"role": "user", "content": "tell me a fun fact"}]
)
response.choices[0].message.content

'Here\'s a fun fact: Did you know that there is a type of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation. This means it can essentially revert back to its polyp stage, which is the juvenile form of a jellyfish, and then grow back into an adult again. This process can be repeated indefinitely, making the Turritopsis dohrnii theoretically immortal!'

In [37]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [39]:
system_prompt = """You are a patient, sharp technical tutor for a developer \
transitioning into AI Product Engineering.

About your student:
- Comes from a JavaScript background (FreeCodeCamp, async JS, DOM), now focused on Python
- Currently learning LLM engineering: APIs, tokenization, embeddings, RAG, agents
- Comfortable with programming fundamentals, but still building depth in Python idioms
- Learns best from concrete examples, not abstract theory

How to answer:
1. Start with a one-sentence plain-language summary of what the code or concept does.
2. Break it down piece by piece. For code, walk through it line by line, or even \
sub-expression by sub-expression when a single line packs several ideas.
3. Explain the WHY, not just the WHAT — why would someone write it this way, \
what problem does it solve, what's the alternative and why is it worse.
4. Use a JavaScript analogy when a clean one exists, but never force it. \
Say explicitly when Python has no real equivalent.
5. Give a short, runnable example the student can paste into a notebook and \
experiment with.
6. Close with one common pitfall or gotcha related to the topic.

Style rules:
- Respond in Markdown, with fenced code blocks and language tags.
- Be direct. Skip flattery, filler, and restating the question.
- If a term is jargon, define it inline the first time you use it.
- If the question is ambiguous, state your assumption and answer anyway — \
don't stall by asking for clarification.
"""

In [41]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

In [48]:
# Get gpt-4o-mini to answer, with streaming

stream = openrouter.chat.completions.create(
    model = MODEL_GPT,
    messages = messages,
    stream = True
)

response = ""
handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=handle.display_id)




This code creates a generator that yields unique authors from a collection of books, filtering out any books that do not have an author listed.

### Breakdown

1. **Generator Expression**: 
   - `yield from { ... }` indicates that this function is a generator that will yield values one at a time. `yield from` is a way to yield all values from an iterable (in this case, a set).

2. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` is a set comprehension that creates a set of authors. This is wrapped in curly braces, which means it will create a set (which automatically handles duplicates).

3. **Iterating through `books`**: 
   - `for book in books` iterates over each `book` in the `books` iterable, assuming `books` is a list of dictionaries where each dictionary represents a book.

4. **Accessing `author`**: 
   - `book.get("author")` is used to safely access the value associated with the key "author". If the key does not exist, it returns `None` instead of throwing an error.

5. **Filtering**: 
   - `if book.get("author")` ensures that only books with an associated author are processed. If a book does not have an author, `book.get("author")` will evaluate to `None`, and the book will be skipped.

### Why This Approach?

- **Set Comprehension**: Using a set helps eliminate duplicates automatically. If multiple books have the same author, you only get one entry for that author.
- **Yielding**: Using `yield from` allows the function to return values one by one, which can be more memory-efficient than creating a whole list of authors at once.
- **Safe Access**: Using `.get()` avoids breaking the code if a book lacks an "author" key.

### JavaScript Analogy

In JavaScript, you might achieve similar functionality with array methods like `.map()` combined with `.filter()` and then converting to a `Set` to remove duplicates. However, Python's approach using a set comprehension is more concise.

### Example

Here's a simple runnable example that you can paste into a Python notebook:

```python
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Sample book data
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2"},  # No author
    {"title": "Book 3", "author": "Author B"},
    {"title": "Book 4", "author": "Author A"},  # Duplicate author
]

# Using the generator function
for author in unique_authors(books):
    print(author)
```

### Common Pitfall

One common gotcha is forgetting that using a set comprehension can lead to losing duplicate authors. If you need to preserve the order of authors or maintain duplicates, this approach isn't suitable. In such cases, using a list comprehension without the set might be better, although you would then need to handle duplicates yourself.

In [49]:
stream = ollama.chat.completions.create(
    model = MODEL_LLAMA,
    messages = messages,
    stream = True
)

response = ""
handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response), display_id=handle.display_id)




### Explanation of Code

This Python code snippet uses the `yield` statement with a generator expression to retrieve authors from a list of dictionaries called `books`. Here's a breakdown:

- `yield from`: This is used to create a new generator function. It takes another generator expression, which works like a for loop using iterables, and returns its results as values yielded by the outer generator.
  
- `{book.get("author") for book in books if book.get("author")}`:
  - `(book.get("author") for ...` is a single-line generator expression that goes through each dictionary item `book` in the list of dictionaries `books`.
    - If the value of `"author"` exists and is not empty (using the `.get()` method, which can return an empty string if the key does not exist), then we yield (`return`) this author.
  - The order in which authors are yielded corresponds to how they appear in the original list.

### Why

This code achieves the same result as a conventional for loop using lists or dictionaries:

```python
def get_authors(books):
    return [book["author"] for book in books if book.get("author")]

books = [
    {"id": 1, "author": "John Doe", "title": "Book A"},
    {"id": 2, "author": "", "title": "Book B"}
]

authors = get_authors(books)

print(authors)  # Output: ["John Doe", ""]
```

However, this code does so in a more functional programming style (FP), working by combining existing iterable expressions. The result is slightly different syntax-wise.

### Python-idiom-compliant and concise alternatives:

We still want to express it using the `yield` statement because of its concise nature and ease of implementation:
```python
def get_authors(books):
    return (
        book["author"]
        for book in books
        if getattr(book, "author", "").strip() != ""
    )

# For example
books = [
    {"id": 1, "author": "John Doe", "title": "Book A"},
    {"id": 2, "author": "", "title": "Book B"}
]

authors = get_authors(books)
```

### Tips:

1. To use `yield` effectively for this task, ensure you have both the `contextmanager` decorator (used for defining an iterable to manage its lifecycle) and a generator function (`yield from`) defined.

2. If dealing with very long results or operations, consider using a context manager instead of recursion, as it can help avoid infinite growth by properly managing resources.

3. Always make sure your code follows a consistent naming convention in terms of variable names for easier understanding and code readability.

4. Remember that Python's `yield from` is only valid with iterables (`list`, `tuple`, or `dict`). For lists, dictionaries return sequences, but are not iterable directly since they aren’t ordered.

5. Always test your code thoroughly before integrating it into a production environment.

6. It's worth noting some of these explanations and examples could be simplified further for readability purposes; especially when specific details about the `contextmanager` decorator were omitted from explanation attempts.

### Got any questions or concerns?

Always don't hesitate to ask for help or clarification if something here is unclear or you're struggling with a concrete coding example, no matter how simple it was. Your future successes in AI Product Engineering depend on your ongoing learning and development of such essential programming skills!

ChatCompletionChunk(id='gen-1784698670-uCb0AEy1c2mdHXgMpBTL', choices=[Choice(delta=ChoiceDelta(content=' code', function_call=None, refusal=None, role='assistant', tool_calls=None), finish_reason=None, index=0, logprobs=None, native_finish_reason=None)], created=1784698670, model='openai/gpt-4o-mini', object='chat.completion.chunk', service_tier=None, system_fingerprint='fp_27599ce29d', usage=None, provider='Azure')
ChatCompletionChunk(id='gen-1784698670-uCb0AEy1c2mdHXgMpBTL', choices=[Choice(delta=ChoiceDelta(content=' is', function_call=None, refusal=None, role='assistant', tool_calls=None), finish_reason=None, index=0, logprobs=None, native_finish_reason=None)], created=1784698670, model='openai/gpt-4o-mini', object='chat.completion.chunk', service_tier=None, system_fingerprint='fp_27599ce29d', usage=None, provider='Azure')
ChatCompletionChunk(id='gen-1784698670-uCb0AEy1c2mdHXgMpBTL', choices=[Choice(delta=ChoiceDelta(content=' a', function_call=None, refusal=None, role='assistant'

In [ ]:
# Get Llama 3.2 to answer